# 📊 07 — Power BI Export & Validation
## Exporting 7 Tables for Power BI Dashboards

Exports all data tables optimized for Power BI import, validates outputs, and provides setup instructions.

---

In [ ]:
import os, sys
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
import config

## 1. Run Export Pipeline

In [ ]:
from src import export_powerbi

export_powerbi.run()

## 2. Validate Exported Files

In [ ]:
print(f"Power BI export directory: {config.OUTPUT_POWERBI}\n")

exported_files = []
for f in sorted(os.listdir(config.OUTPUT_POWERBI)):
    if f.endswith('.csv'):
        path = os.path.join(config.OUTPUT_POWERBI, f)
        df = pd.read_csv(path)
        size_kb = os.path.getsize(path) / 1024
        exported_files.append({
            'File': f,
            'Rows': f'{len(df):,}',
            'Columns': len(df.columns),
            'Size (KB)': f'{size_kb:.0f}',
        })
        print(f"✓ {f}: {len(df):,} rows × {len(df.columns)} cols ({size_kb:.0f} KB)")

print(f"\nTotal files: {len(exported_files)}")

## 3. Preview Each Table

In [ ]:
for f in sorted(os.listdir(config.OUTPUT_POWERBI)):
    if f.endswith('.csv'):
        print(f"\n{'='*60}")
        print(f"  {f}")
        print(f"{'='*60}")
        df = pd.read_csv(os.path.join(config.OUTPUT_POWERBI, f))
        display(df.head(3))

## 4. Power BI Relationship Diagram

```
┌──────────────────┐     ┌─────────────────────────┐
│  date_dimension  │     │   transactions_summary   │
│  (date)          │     │   (customer_id,           │
│                  │     │    year_month)            │
└────────┬─────────┘     └────────────┬──────────────┘
         │                            │
         │ 1:M                   M:1  │
         │                            │
         └───────────┐    ┌───────────┘
                     │    │
              ┌──────┴────┴──────┐
              │  customer_master │ ← Primary Table
              │  (customer_id)   │
              └──┬────┬────┬─────┘
                 │    │    │
            1:1  │    │    │ M:1
                 │    │    │
     ┌───────────┘    │    └───────────┐
     │                │                │
┌────┴──────────┐ ┌───┴───────────┐ ┌──┴──────────────┐
│  retention_   │ │ geographic_   │ │ segment_summary │
│  decisions    │ │ summary       │ │ (rfm_segment)   │
│  (top 100)    │ │               │ │                 │
└───────────────┘ └───────────────┘ └─────────────────┘

              ┌─────────────────────┐
              │ category_performance│ (standalone)
              └─────────────────────┘
```

## 5. Quick Power BI Setup Instructions

1. **Open Power BI Desktop** → Get Data → CSV → Select all files from `outputs/powerbi/`
2. **Create Relationships**:
   - `customer_master[customer_id]` → `transactions_summary[customer_id]` (1:M)
   - `customer_master[customer_id]` → `retention_decisions[customer_id]` (1:1)
   - `customer_master[rfm_segment]` → `segment_summary[rfm_segment]` (M:1)
3. **See `docs/POWERBI_DASHBOARD_GUIDE.md`** for complete page layouts, DAX measures, and design specifications.

---
**✅ All exports complete!** Open Power BI Desktop and import from `outputs/powerbi/`.